# AFML Part 1 — Denoising Corrupted Weights¶

TEAM ID and Name: 26, Biased Bozos
TEAM MEMBERS:

    Arvind Ajay Bharadwaj - PES1UG24AM052
    Atharva Gore - PES1UG24AM346
    Bhooshan Deshpande - PES1UG24AM351
    Deeptanshu Kumar - PES1UG24AM357

The final cell writes the Part 2 outputs:

    submission_part2.csv

In this part, we will use our restored weights from **Part 1** to initialize an **Encoder-Decoder Transformer**, fine-tune it on our team's parallel corpus, and generate English translations for test queries.

### What is in this Notebook?
This notebook is completely self-contained with **zero external script dependencies**. Everything you need is in this single notebook:
1. **Self-Contained Tokenizer**: Word-level tokenizer with vocabularies of both asdfghjkl and English.
2. **TranslationTransformer**: Complete 2.96M-parameter PyTorch Transformer with weight serialization and autoregressive decoding.
3. **Weight Loading**: Injects your Part 1 restored weights (`denoised_test_weights.npy` or `submission_part1.csv`).
4. **Low-Resource Fine-Tuning**: Supervised calibration loop on `train_part2.csv` (or `train_parallel_corpus.csv`).
5. **Inference & Submission**: Generates predictions for `test_part2.csv` and outputs `submission_part2.csv`.


In [52]:
# Cell 1: Environment & Imports
import os
import re
import sys
import json
import math
import zlib
import base64
import random
from copy import deepcopy
from typing import List, Dict, Union, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device   : {device}')


Active Device   : cuda


In [53]:
# Cell 2: Kaggle Dataset Path Auto-Discovery
def find_data_file(filename: str, fallbacks: Optional[List[str]] = None) -> str:
    """Search local directory and Kaggle /kaggle/input paths."""
    candidates = [filename] + (fallbacks or [])
    # 1. Check local working directory
    for cand in candidates:
        if os.path.exists(cand):
            return cand
    # 2. Check Kaggle input mount points
    kaggle_dir = '/kaggle/input'
    if os.path.exists(kaggle_dir):
        for root, _, files in os.walk(kaggle_dir):
            for cand in candidates:
                if cand in files:
                    return os.path.join(root, cand)
    return candidates[0]

TRAIN_CORPUS_FILE = "/kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/train_parallel_corpus.csv"
TEST_QUERIES_FILE = "/kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/test_queries.csv"
RESTORED_WEIGHTS_FILE = "/kaggle/input/datasets/deeptanshukumar/afml-denoiser/denoised_test_weights.npy"

print(f'Parallel Corpus  : {TRAIN_CORPUS_FILE} (Found: {os.path.exists(TRAIN_CORPUS_FILE)})')
print(f'Test Queries     : {TEST_QUERIES_FILE} (Found: {os.path.exists(TEST_QUERIES_FILE)})')
print(f'Restored Weights : {RESTORED_WEIGHTS_FILE} (Found: {os.path.exists(RESTORED_WEIGHTS_FILE)})')


Parallel Corpus  : /kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/train_parallel_corpus.csv (Found: True)
Test Queries     : /kaggle/input/competitions/afml-orange-problem-denoising-transformer-weights/26_Biased_Bozos/test_queries.csv (Found: True)
Restored Weights : /kaggle/input/datasets/deeptanshukumar/afml-denoiser/denoised_test_weights.npy (Found: True)


In [54]:
# Cell 3: Vocabulary & Tokenizer Setup
_EMBEDDED_VOCAB_ASDF_B64 = "..."
_EMBEDDED_VOCAB_ENG_B64 = "..."

SPECIAL_TOKENS = {
    "<pad>": 0,
    "<bos>": 1,
    "<eos>": 2,
    "<unk>": 3,
}

class SimpleTokenizer:
    """
    Word and sequence tokenizer supporting both 'asdfghjkl' and English text.
    Handles text sanitization, whitespace segmentation, and vocabulary indexing.
    """
    def __init__(self, is_source: bool = True, vocab_file: Optional[str] = None):
        self.is_source = is_source
        self.pad_token = '<pad>'
        self.bos_token = '<bos>'
        self.eos_token = '<eos>'
        self.unk_token = '<unk>'

        self.special_tokens = dict(SPECIAL_TOKENS)
        self.word2idx: Dict[str, int] = dict(self.special_tokens)

        default_file = 'vocab_asdf.json' if is_source else 'vocab_eng.json'
        resolved_file = vocab_file or find_data_file(default_file, [os.path.join('..', default_file)])

        if os.path.exists(resolved_file):
            with open(resolved_file, 'r', encoding='utf-8') as f:
                loaded = json.load(f)
        else:
            b64_str = _EMBEDDED_VOCAB_ASDF_B64 if is_source else _EMBEDDED_VOCAB_ENG_B64
            decompressed = zlib.decompress(base64.b64decode(b64_str.encode('ascii'))).decode('utf-8')
            loaded = json.loads(decompressed)

        control_keys = {"<pad>", "<unk>", "<bos>", "<eos>"}
        for word, idx in loaded.items():
            if word not in control_keys and word not in self.special_tokens:
                self.word2idx[word] = idx

        self.idx2word: Dict[int, str] = {int(idx): word for word, idx in self.word2idx.items()}

    @property
    def pad_id(self) -> int: return self.word2idx.get(self.pad_token, 0)
    @property
    def unk_id(self) -> int: return self.word2idx.get(self.unk_token, 3)
    @property
    def bos_id(self) -> int: return self.word2idx.get(self.bos_token, 1)
    @property
    def eos_id(self) -> int: return self.word2idx.get(self.eos_token, 2)
    @property
    def vocab_size(self) -> int: return len(self.word2idx)

    def tokenize(self, text: str) -> List[str]:
        if self.is_source:
            return [t.strip() for t in str(text).strip().split() if t.strip()]
        else:
            tokens = []
            for t in str(text).strip().split():
                clean = re.sub(r"[^\w\'-]", "", t, flags=re.UNICODE).lower()
                if clean:
                    tokens.append(clean)
            return tokens

    def encode(self, text: str, add_special_tokens: bool = True) -> List[int]:
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(t, self.unk_id) for t in tokens]
        if add_special_tokens:
            return [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids: List[int], skip_special_tokens: bool = True) -> str:
        words = []
        special_ids = {self.pad_id, self.bos_id, self.eos_id}
        for i in ids:
            if skip_special_tokens and i in special_ids:
                if i == self.eos_id:
                    break
                continue
            word = self.idx2word.get(i, self.unk_token)
            if word and word != self.unk_token:
                words.append(word)
        return ' '.join(words)

src_tok = SimpleTokenizer(is_source=True)
tgt_tok = SimpleTokenizer(is_source=False)
print(f'Vocab sizes: source={src_tok.vocab_size:,}, target={tgt_tok.vocab_size:,}')


Vocab sizes: source=5,897, target=5,932


In [55]:
# Cell 6: Dataset & Low-Resource Fine-Tuning
class TranslationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, src_tok: SimpleTokenizer, tgt_tok: SimpleTokenizer, max_len: int = 35):
        # Auto-detect source and target columns
        src_col = 'encoded_text' if 'encoded_text' in df.columns else ('asdfghjkl' if 'asdfghjkl' in df.columns else df.columns[1])
        tgt_col = 'text' if 'text' in df.columns else ('english' if 'english' in df.columns else df.columns[2])

        self.data = []
        for _, row in df.iterrows():
            src_ids = src_tok.encode(row[src_col])[:max_len]
            tgt_ids = tgt_tok.encode(row[tgt_col])[:max_len]
            self.data.append((src_ids, tgt_ids))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn(batch, src_pad, tgt_pad):
    src_list, tgt_list = zip(*batch)
    ms, mt = max(len(s) for s in src_list), max(len(t) for t in tgt_list)
    p_src = [s + [src_pad] * (ms - len(s)) for s in src_list]
    p_tgt = [t + [tgt_pad] * (mt - len(t)) for t in tgt_list]
    return torch.tensor(p_src, dtype=torch.long), torch.tensor(p_tgt, dtype=torch.long)


In [56]:
# Cell 4: TranslationTransformer Architecture
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]


class TranslationTransformer(nn.Module):
    """
    Encoder-Decoder Transformer (2.96M parameters).
    Matches the pre-trained weights from Part 1.
    """
    def __init__(
        self,
        src_vocab_size: int = 5947,
        tgt_vocab_size: int = 5982,
        d_model: int = 128,
        nhead: int = 4,
        num_encoder_layers: int = 2,
        num_decoder_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def generate_square_subsequent_mask(self, sz: int, device: torch.device) -> torch.Tensor:
        mask = (torch.triu(torch.ones(sz, sz, device=device)) == 1).transpose(0, 1)
        return mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, src_pad_idx: int = 0, tgt_pad_idx: int = 0) -> torch.Tensor:
        device = src.device
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1), device)
        src_padding_mask = (src == src_pad_idx)
        tgt_padding_mask = (tgt == tgt_pad_idx)

        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
        )
        return self.fc_out(out)

    def load_flattened_weights(self, flat_weights):
        """Inject flattened 1D array of weights into model parameters."""
        if isinstance(flat_weights, np.ndarray):
            flat_weights = torch.tensor(flat_weights, dtype=torch.float32)

        offset = 0
        total_avail = flat_weights.numel()
        with torch.no_grad():
            for name, param in self.named_parameters():
                n_el = param.numel()
                if offset + n_el <= total_avail:
                    chunk = flat_weights[offset:offset + n_el].view_as(param)
                    param.copy_(chunk)
                    offset += n_el
                else:
                    break

    def generate(
        self,
        src: torch.Tensor,
        max_len: int = 25,
        bos_idx: int = 1,
        eos_idx: int = 2,
        repetition_penalty: float = 1.25,
        no_repeat_ngram_size: int = 2,
    ) -> torch.Tensor:
        self.eval()
        device = src.device
        batch_size = src.size(0)
        tgt = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)

        with torch.no_grad():
            for _ in range(max_len):
                logits = self.forward(src, tgt)[:, -1, :]

                if repetition_penalty != 1.0:
                    for b in range(batch_size):
                        for tid in set(tgt[b].tolist()):
                            if logits[b, tid] > 0:
                                logits[b, tid] /= repetition_penalty
                            else:
                                logits[b, tid] *= repetition_penalty

                if no_repeat_ngram_size > 0 and tgt.size(1) >= no_repeat_ngram_size:
                    for b in range(batch_size):
                        seq = tgt[b].tolist()
                        cur_prefix = tuple(seq[-(no_repeat_ngram_size - 1):])
                        for i in range(len(seq) - no_repeat_ngram_size + 1):
                            if tuple(seq[i:i + no_repeat_ngram_size - 1]) == cur_prefix:
                                banned = seq[i + no_repeat_ngram_size - 1]
                                logits[b, banned] = float('-inf')

                next_token = torch.argmax(logits, dim=-1, keepdim=True)
                tgt = torch.cat([tgt, next_token], dim=1)

                if batch_size == 1:
                    if next_token.item() == eos_idx:
                        break
                elif (tgt == eos_idx).any(dim=1).all():
                    break

        return tgt

model = TranslationTransformer(5947, 5982, d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2, dim_feedforward=256).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model Initialized | Total Parameters: {n_params:,}')


Model Initialized | Total Parameters: 2,961,630


In [57]:
# Cell 5: Load Restored Weights from Part 1
def load_weights_file(path: str) -> np.ndarray:
    if path.endswith('.npy'):
        return np.load(path).astype(np.float32).reshape(-1)
    else:
        df = pd.read_csv(path)
        col = 'weight' if 'weight' in df.columns else df.columns[1] if len(df.columns) > 1 else df.columns[0]
        return df[col].to_numpy(dtype=np.float32).reshape(-1)

if os.path.exists(RESTORED_WEIGHTS_FILE):
    flat_weights = load_weights_file(RESTORED_WEIGHTS_FILE)
    print(f'Loading {len(flat_weights):,} parameters from {RESTORED_WEIGHTS_FILE}...')
    model.load_flattened_weights(flat_weights)
    print('Restored weights successfully loaded into TranslationTransformer!')
else:
    print(f'Notice: {RESTORED_WEIGHTS_FILE} not found. Running model with initial weights.')


Loading 2,961,630 parameters from /kaggle/input/datasets/deeptanshukumar/afml-denoiser/denoised_test_weights.npy...
Restored weights successfully loaded into TranslationTransformer!


# 05. Validation experiment

We use a fixed 240/60 split to compare the decoding ideas before touching the full 300-pair corpus.
This lets us check whether the extra inference logic actually helps on held-out examples.


In [58]:
# Validation helpers
diag_df = pd.read_csv(TRAIN_CORPUS_FILE).reset_index(drop=True)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(diag_df))
split_at = int(0.80 * len(diag_df))

diag_train_df = diag_df.iloc[perm[:split_at]].reset_index(drop=True)
diag_val_df = diag_df.iloc[perm[split_at:]].reset_index(drop=True)


def _target_tokens(text):
    return tgt_tok.tokenize(text)


def _source_tokens(text):
    return src_tok.tokenize(text)


def token_metrics(pred_token_lists, ref_token_lists):
    total_ref = 0
    matched = 0
    exact = 0

    for pred, ref in zip(pred_token_lists, ref_token_lists):
        total_ref += len(ref)
        matched += sum(p == r for p, r in zip(pred, ref))
        exact += int(pred == ref)

    token_acc = matched / total_ref if total_ref else 0.0
    exact_rate = exact / len(ref_token_lists) if ref_token_lists else 0.0
    return token_acc, exact_rate


def build_token_dictionary(df):
    """Majority source-token -> target-token mapping using aligned rows."""
    counts = {}
    aligned_rows = 0
    skipped_rows = 0

    for _, row in df.iterrows():
        source = _source_tokens(row["asdfghjkl"])
        target = _target_tokens(row["english"])

        if len(source) != len(target):
            skipped_rows += 1
            continue

        aligned_rows += 1
        for s_tok, t_tok in zip(source, target):
            counts.setdefault(s_tok, {})
            counts[s_tok][t_tok] = counts[s_tok].get(t_tok, 0) + 1

    mapping = {
        s_tok: max(target_counts.items(), key=lambda kv: kv[1])[0]
        for s_tok, target_counts in counts.items()
    }

    ambiguous = [
        (s_tok, sorted(target_counts.items(), key=lambda kv: kv[1], reverse=True)[:3])
        for s_tok, target_counts in counts.items()
        if len(target_counts) > 1
    ]

    return mapping, aligned_rows, skipped_rows, ambiguous


def dictionary_translate(text, mapping):
    source = _source_tokens(text)
    pred = [mapping.get(tok, "<unk>") for tok in source]
    known = sum(tok in mapping for tok in source)
    return pred, known, len(source)


def translate_df_with_model(eval_model, df):
    eval_model.eval()
    predictions = []

    with torch.no_grad():
        for _, row in df.iterrows():
            src_ids = src_tok.encode(row["asdfghjkl"], add_special_tokens=True)
            src_t = torch.tensor([src_ids], dtype=torch.long, device=device)

            gen_ids = eval_model.generate(
                src_t,
                max_len=25,
                bos_idx=tgt_tok.bos_id,
                eos_idx=tgt_tok.eos_id,
                repetition_penalty=1.25,
                no_repeat_ngram_size=2,
            )[0].cpu().tolist()

            predictions.append(tgt_tok.decode(gen_ids, skip_special_tokens=True))

    return predictions


def length_correct_transformer_words(transformer_words, expected_len):
    transformer_words = list(transformer_words)
    if len(transformer_words) >= expected_len:
        return transformer_words[:expected_len]
    return transformer_words + ["<unk>"] * (expected_len - len(transformer_words))


def hybrid_translate_from_words(source_text, transformer_text, mapping):
    source_words = _source_tokens(source_text)
    transformer_words = length_correct_transformer_words(
        _target_tokens(transformer_text), len(source_words)
    )

    return [
        mapping[source_word] if source_word in mapping else transformer_words[i]
        for i, source_word in enumerate(source_words)
    ]


def _apply_repetition_penalty_to_logits(logits, generated_ids, penalty=1.25):
    logits = logits.clone()
    if penalty == 1.0:
        return logits

    for tid in set(generated_ids):
        if 0 <= tid < logits.numel():
            logits[tid] = (
                logits[tid] / penalty
                if logits[tid] > 0
                else logits[tid] * penalty
            )
    return logits


def constrained_hybrid_translate(eval_model, source_text, mapping, repetition_penalty=1.25):
    """Decode one English token per source token."""
    eval_model.eval()
    source_words = _source_tokens(source_text)

    src_ids = src_tok.encode(source_text, add_special_tokens=True)
    src_t = torch.tensor([src_ids], dtype=torch.long, device=device)
    generated = [tgt_tok.bos_id]

    with torch.no_grad():
        for source_word in source_words:
            if source_word in mapping:
                target_word = mapping[source_word]
                next_id = tgt_tok.word2idx.get(target_word, tgt_tok.unk_id)
            else:
                tgt_t = torch.tensor([generated], dtype=torch.long, device=device)
                logits = eval_model(
                    src_t,
                    tgt_t,
                    src_pad_idx=src_tok.pad_id,
                    tgt_pad_idx=tgt_tok.pad_id,
                )[0, -1, :]

                logits = _apply_repetition_penalty_to_logits(
                    logits, generated, penalty=repetition_penalty
                )

                banned = {
                    tgt_tok.pad_id,
                    tgt_tok.bos_id,
                    tgt_tok.eos_id,
                    tgt_tok.unk_id,
                }
                for tid in banned:
                    if 0 <= tid < logits.numel():
                        logits[tid] = float('-inf')

                next_id = int(torch.argmax(logits).item())

            generated.append(next_id)

    generated.append(tgt_tok.eos_id)
    return tgt_tok.decode(generated, skip_special_tokens=True), generated


### 05.A. Dictionary baseline

The training pairs show a mostly position-preserving source-token → English-token mapping,
so a simple dictionary gives us a useful baseline for the validation experiment.


In [59]:
dictionary, aligned_rows, skipped_rows, ambiguous = build_token_dictionary(diag_train_df)

dict_preds, dict_refs = [], []
for _, row in diag_val_df.iterrows():
    pred, _, _ = dictionary_translate(row["asdfghjkl"], dictionary)
    dict_preds.append(pred)
    dict_refs.append(_target_tokens(row["english"]))

dict_token_acc, dict_exact = token_metrics(dict_preds, dict_refs)


### 05.B. Restored Transformer

Next we test the Part 1 weights directly, before any low-resource fine-tuning.
This tells us how much translation ability survived the denoising step on its own.


In [60]:
restored_preds = translate_df_with_model(model, diag_val_df)
restored_pred_tokens = [_target_tokens(x) for x in restored_preds]
restored_ref_tokens = [_target_tokens(x) for x in diag_val_df["english"].tolist()]

restored_token_acc, restored_exact = token_metrics(
    restored_pred_tokens, restored_ref_tokens
)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


### 05.C. Fine-tuned Transformer

We fine-tune a separate copy on only the 240 training examples, so the restored model remains untouched.
The validation run uses the same setting we plan to use for the final run: **40 epochs, learning rate 5e-5**.


In [61]:
VAL_EPOCHS = 40
VAL_LR = 5e-5

val_model = deepcopy(model).to(device)
val_ds = TranslationDataset(diag_train_df, src_tok, tgt_tok, max_len=35)
val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, src_tok.pad_id, tgt_tok.pad_id),
)

val_optimizer = torch.optim.AdamW(
    val_model.parameters(),
    lr=VAL_LR,
    weight_decay=1e-4,
)
val_criterion = nn.CrossEntropyLoss(
    ignore_index=tgt_tok.pad_id,
    label_smoothing=0.05,
)

for ep in range(1, VAL_EPOCHS + 1):
    total_loss = 0.0
    val_model.train()

    for src_b, tgt_b in val_loader:
        src_b, tgt_b = src_b.to(device), tgt_b.to(device)
        out = val_model(
            src_b,
            tgt_b[:, :-1],
            src_pad_idx=src_tok.pad_id,
            tgt_pad_idx=tgt_tok.pad_id,
        )
        loss = val_criterion(
            out.reshape(-1, out.size(-1)),
            tgt_b[:, 1:].reshape(-1),
        )

        val_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(val_model.parameters(), 1.0)
        val_optimizer.step()

        total_loss += loss.item() * len(src_b)


### 05.D. Length correction

Raw autoregressive generation sometimes produces too many or too few words.
Since the corpus is position-preserving, we test a simple fix: force the output length to match the source length.


In [62]:
ft_preds = translate_df_with_model(val_model, diag_val_df)

ft_pred_tokens = [_target_tokens(x) for x in ft_preds]
ft_token_acc, ft_exact = token_metrics(ft_pred_tokens, dict_refs)

ft_len_preds = []
for i, (_, row) in enumerate(diag_val_df.iterrows()):
    source_len = len(_source_tokens(row["asdfghjkl"]))
    ft_words = length_correct_transformer_words(
        _target_tokens(ft_preds[i]),
        source_len,
    )
    ft_len_preds.append(ft_words)

ft_len_token_acc, ft_len_exact = token_metrics(ft_len_preds, dict_refs)


### 05.E. Post-hoc hybrid

Here the dictionary handles known source tokens, while the fine-tuned Transformer supplies predictions for unknown ones.
Length correction is applied first so the two sources of information stay position-aligned.


In [63]:
hybrid_preds = []
for i, (_, row) in enumerate(diag_val_df.iterrows()):
    hybrid_preds.append(
        hybrid_translate_from_words(
            row["asdfghjkl"],
            ft_preds[i],
            dictionary,
        )
    )

hybrid_token_acc, hybrid_exact = token_metrics(hybrid_preds, dict_refs)


### 05.F. Position-constrained hybrid

The previous hybrid repairs alignment after the Transformer has generated a whole sentence.
This version keeps source and target positions synchronized from the start: known tokens are forced, unknown tokens are chosen by the Transformer.


In [64]:
constrained_preds = []
for _, row in diag_val_df.iterrows():
    pred_text, _ = constrained_hybrid_translate(
        val_model,
        row["asdfghjkl"],
        dictionary,
        repetition_penalty=1.25,
    )
    constrained_preds.append(_target_tokens(pred_text))

constrained_token_acc, constrained_exact = token_metrics(
    constrained_preds, dict_refs
)

validation_results = pd.DataFrame({
    "Method": [
        "Dictionary baseline",
        "Restored Transformer",
        "Fine-tuned Transformer",
        "Length-corrected Transformer",
        "Post-hoc hybrid",
        "Position-constrained hybrid",
    ],
    "Token accuracy": [
        dict_token_acc,
        restored_token_acc,
        ft_token_acc,
        ft_len_token_acc,
        hybrid_token_acc,
        constrained_token_acc,
    ],
    "Exact match": [
        dict_exact,
        restored_exact,
        ft_exact,
        ft_len_exact,
        hybrid_exact,
        constrained_exact,
    ],
})
validation_results.style.format({
    "Token accuracy": "{:.2%}",
    "Exact match": "{:.2%}",
})


,Method,Token accuracy,Exact match
0,Dictionary baseline,78.44%,6.67%
1,Restored Transformer,31.44%,3.33%
2,Fine-tuned Transformer,62.07%,25.00%
3,Length-corrected Transformer,62.07%,36.67%
4,Post-hoc hybrid,88.17%,41.67%
5,Position-constrained hybrid,92.38%,50.00%


In [65]:
del val_model, val_optimizer, val_loader, val_ds
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Validation result used for the final setup

Earlier checks with **12, 20, 40, and 60 epochs at 5e-5** showed that 40 epochs gave the strongest held-out result.
So we lock **40 epochs / 5e-5** and carry the position-constrained hybrid into final inference.


# 06. Final training: 40 epochs / 5e-5

Now we fine-tune the original restored-weight model on all 300 sentence pairs.
The validation model was a separate copy, so `model` still starts from the denoised Part 1 weights.


In [66]:
if os.path.exists(TRAIN_CORPUS_FILE):
    train_df = pd.read_csv(TRAIN_CORPUS_FILE)
    train_ds = TranslationDataset(train_df, src_tok, tgt_tok)
    train_loader = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, src_tok.pad_id, tgt_tok.pad_id)
    )

    ft_epochs = 40
    ft_lr = 5e-5
    optimizer = torch.optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_tok.pad_id, label_smoothing=0.05)

    model.train()
    print(f'Fine-tuning on {len(train_ds)} sentence pairs for {ft_epochs} epochs...')
    for ep in range(1, ft_epochs + 1):
        total_loss = 0.0
        for src_b, tgt_b in train_loader:
            src_b, tgt_b = src_b.to(device), tgt_b.to(device)
            out = model(src_b, tgt_b[:, :-1], src_pad_idx=src_tok.pad_id, tgt_pad_idx=tgt_tok.pad_id)
            loss = criterion(out.reshape(-1, out.size(-1)), tgt_b[:, 1:].reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(src_b)

        if ep in [1, 10, 20, 30, ft_epochs]:
            print(f'  Epoch {ep:02d}/{ft_epochs:02d} | Cross-Entropy Loss: {total_loss / len(train_ds):.4f}')
    print('Fine-tuning completed!')
else:
    print('Skipping fine-tuning: corpus file not found.')


Fine-tuning on 300 sentence pairs for 40 epochs...
  Epoch 01/40 | Cross-Entropy Loss: 1.9798


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


  Epoch 10/40 | Cross-Entropy Loss: 0.8764
  Epoch 20/40 | Cross-Entropy Loss: 0.7445
  Epoch 30/40 | Cross-Entropy Loss: 0.7120
  Epoch 40/40 | Cross-Entropy Loss: 0.6979
Fine-tuning completed!


# 07. Final constrained-hybrid inference

For each source position, known tokens use the learned dictionary mapping.
Unknown tokens are predicted by the fine-tuned Transformer, with exactly one output token per source token.


In [67]:
if os.path.exists(TEST_QUERIES_FILE):
    test_df = pd.read_csv(TEST_QUERIES_FILE)
    src_col = 'encoded_text' if 'encoded_text' in test_df.columns else ('asdfghjkl' if 'asdfghjkl' in test_df.columns else test_df.columns[1])
    id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]

    full_dictionary, _, _, full_ambiguous = build_token_dictionary(train_df)

    predictions = []
    for _, row in test_df.iterrows():
        pred_text, _ = constrained_hybrid_translate(
            model,
            row[src_col],
            full_dictionary,
            repetition_penalty=1.25,
        )
        predictions.append(pred_text)

    print(f'Translated {len(predictions)} test queries using position-constrained hybrid decoding.')
else:
    print('Test queries file not found.')


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Translated 200 test queries using position-constrained hybrid decoding.


# 08. Submission

The final CSV uses the competition's `id, translation` format.
We save only the final position-constrained hybrid predictions.


In [68]:
if os.path.exists(TEST_QUERIES_FILE):
    sub_df = pd.DataFrame({
        'id': test_df[id_col],
        'translation': predictions
    })
    sub_df.to_csv('submission_part2.csv', index=False)
    print(f'Successfully exported {len(predictions)} predictions to submission_part2.csv!')
else:
    print('Submission skipped: test queries file not found.')


Successfully exported 200 predictions to submission_part2.csv!
